In [ ]:
!pip install --no-deps bitsandbytes accelerate xformers peft trl triton cut_cross_entropy unsloth_zoo unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments
from trl import SFTConfig
from peft import LoraConfig

In [ ]:
medical_prompt = """### Instruction:
You are a medical scribe. Below is a transcript of a doctor-patient encounter.
Summarize the conversation into a structured SOAP note (Subjective, Objective, Assessment, Plan).

### Transcript:
{}

### Response:
{}"""

In [ ]:
# Load the dataset
train_dataset = load_dataset("mkieffer/ACI-Bench", "aci", split="train")

In [ ]:
# Prepare dataset
EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples):
    # "dialogue" and "note" seem to be the correct column names based on ACI-Bench usage
    inputs  = examples["dialogue"]
    outputs = examples["note"]

    texts = []
    for dialogue, note in zip(inputs, outputs):
        # Format → append EOS (very important for causal LM)
        full_text = medical_prompt.format(dialogue, note) + EOS_TOKEN
        texts.append(full_text)

    return {"text": texts}


# Apply once (batched is faster)
train_dataset = train_dataset.map(
    formatting_prompts_func,
    batched=True,
    remove_columns=train_dataset.column_names,   # clean up original columns
    desc="Formatting medical dialogue → note examples"
)

Map:   0%|          | 0/35 [00:00<?, ? examples/s]

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

model_id = "google/medgemma-4b-it"

# Check if GPU supports bfloat16
if torch.cuda.get_device_capability()[0] < 8:
    raise ValueError("GPU does not support bfloat16, please use a GPU that supports bfloat16.")

model_kwargs = dict(
    attn_implementation="eager",
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

model_kwargs["quantization_config"] = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=model_kwargs["torch_dtype"],
    bnb_4bit_quant_storage=model_kwargs["torch_dtype"],
)

model = AutoModelForImageTextToText.from_pretrained(model_id, **model_kwargs)
processor = AutoProcessor.from_pretrained(model_id)

# Use right padding to avoid issues during training
processor.tokenizer.padding_side = "right"

In [ ]:
from peft import LoraConfig

peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.05,
    r=16,
    bias="none",
    target_modules="all-linear",
    task_type="CAUSAL_LM",
    modules_to_save=[
        "lm_head",
        "embed_tokens",
    ],
)

In [ ]:
from typing import Any, Dict, List
import torch

def collate_fn(examples: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
    """
    Collate function for pure causal LM (next-token prediction).
    Expects dataset with 'text' field already formatted (prompt + completion + EOS).
    """
    # Extract texts
    texts = [ex["text"] for ex in examples]

    # Tokenize + pad
    batch = tokenizer(
        texts,
        padding=True,           # pad to longest in batch
        truncation=True,
        max_length=2048,        # ← adjust to your model's context & GPU memory
        return_tensors="pt",
    )

    # For causal LM: labels = input_ids, but ignore padding in loss
    labels = batch["input_ids"].clone()

    # Standard: mask pad tokens → -100 (CrossEntropy ignores -100)
    if tokenizer.pad_token_id is not None:
        labels[labels == tokenizer.pad_token_id] = -100

    batch["labels"] = labels

    return batch

In [ ]:
from trl import SFTConfig

num_train_epochs = 1
learning_rate = 2e-4

args = SFTConfig(
    output_dir="medgemma-4b-it-sft-lora-crc100k",            # Directory and Hub repository id to save the model to
    num_train_epochs=num_train_epochs,                       # Number of training epochs
    per_device_train_batch_size=4,                           # Batch size per device during training
    per_device_eval_batch_size=4,                            # Batch size per device during evaluation
    gradient_accumulation_steps=4,                           # Number of steps before performing a backward/update pass
    gradient_checkpointing=True,                             # Enable gradient checkpointing to reduce memory usage
    optim="adamw_torch_fused",                               # Use fused AdamW optimizer for better performance
    logging_steps=50,                                        # Number of steps between logs
    save_strategy="epoch",                                   # Save checkpoint every epoch
    eval_strategy="steps",                                   # Evaluate every `eval_steps`
    eval_steps=50,                                           # Number of steps between evaluations
    learning_rate=learning_rate,                             # Learning rate based on QLoRA paper
    bf16=True,                                               # Use bfloat16 precision
    max_grad_norm=0.3,                                       # Max gradient norm based on QLoRA paper
    warmup_ratio=0.03,                                       # Warmup ratio based on QLoRA paper
    lr_scheduler_type="linear",                              # Use linear learning rate scheduler
    push_to_hub=True,                                        # Push model to Hub
    report_to="tensorboard",                                 # Report metrics to tensorboard
    gradient_checkpointing_kwargs={"use_reentrant": False},  # Set gradient checkpointing to non-reentrant to avoid issues
    dataset_kwargs={"skip_prepare_dataset": True},           # Skip default dataset preparation to preprocess manually
    remove_unused_columns = False,                           # Columns are unused for training but needed for data collator
    label_names=["labels"],                                  # Input keys that correspond to the labels
)

In [ ]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=data["train"],
    eval_dataset=data["validation"].shuffle().select(range(200)),  # Use subset of validation set for faster run
    peft_config=peft_config,
    processing_class=processor,
    data_collator=collate_fn,
)

In [ ]:
trainer.save_model()